# 03. Scenario 2 — "My overseas transfer has not arrived" (tool retry loop)

The second support scenario. This time the question contains enough detail (transfer reference and amount), but **the first tool returns a system error (tool_error)**. The agent reads the error at D6 (tool-result review), decides `try_alternative`, and runs the recursive loop that **retries with a different tool**.

Expected path:
```
D1 -> D2(transfers) -> D5(info_sufficient) -> D3(track_transfer) -> tool_error
-> D6(try_alternative) -> D3(alternative tool, failed one excluded) -> success -> D6(goal_met)
-> summary -> D7(no_risk) -> D4(resolve) -> final reply
```

This is the everyday "transient failure -> automatic recovery" flow of production systems. With the retry loop the decision count grows to 9, which makes the cumulative effect of decision latency easier to see.


In [ ]:
import os
os.environ["DMWS_LANG"] = "en"   # localize helper output/graph captions (set BEFORE importing ws_common)
import sys; from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))  # ws_common path (relative inside the repo)
import ws_common as W
graphs, client = W.make_graphs()
sc = W.SCENARIOS[2]
print("Scenario:", sc["title_en"])
print("Customer question (Korean input):", sc["question"])
print("English gloss:", sc["question_en"])
print("script (pre-set for reproducibility):", sc["script"])   # tool_outcomes: [tool_error, success]


## Where this scenario sits in the full workflow

The path this scenario takes (red borders): straight to the tool with no clarification, but the first tool fails, so the **self-loop on tool_step (retry)** runs once. Watch the same node execute twice in the STEP output.

Assigned team (red box): **transfers (transfers/remittance)** — tracking an overseas remittance, handled with transfers team tools (track_transfer etc.).
> Note: this diagram is not auto-generated from the running LangGraph; it is a **visualization-only drawing** rendered from the caption dictionaries in `ws_common.py`, keyed by node name (details: `[EN]_01_build_agent.ipynb`).


In [ ]:
W.draw_graph(highlight=W.SCENARIO_PATHS[2], show_teams=True, width=760)

## (1) Run with Haiku 4.5 decisions

In the STEP output, confirm that tool_step appears **twice** (failure -> retry with an alternative tool), and that the second D3 decision receives the "Already tried (failed)" information.


In [ ]:
res_haiku = W.run_verbose(graphs["Haiku"], sc["question"], sc["script"], "Haiku")

## (2) Run with SGLang (Qwen3-VL-8B) decisions


In [ ]:
res_sglang = W.run_verbose(graphs["SGLang"], sc["question"], sc["script"], "SGLang")

## (3) Execution time comparison


In [ ]:
W.compare(res_haiku, res_sglang)

**What to look for**
- D6 is a decision that **reads the tool's JSON result and judges the situation** — harder than classification, yet check whether both backends reached the same conclusions (try_alternative -> goal_met).
- With 9 decisions, the Haiku arm's cumulative decision time reaches 5-6 seconds. This is why the decision layer is worth more the more loops an agent has.
- Next: `[EN]_04_scenario3_risk_kyc.ipynb` — the fraud-monitoring flag and the KYC re-verification sub-flow.
